# Dataset Splitting in Machine Learning

This notebook explains the essential practice of splitting datasets in machine learning projects. We'll cover:

1. Basic train-test splits
2. Train-validation-test splits
3. K-fold cross-validation
4. Handling imbalanced datasets

Throughout this notebook, we'll use datasets from scikit-learn to demonstrate these concepts with practical examples.

## Why Split Your Dataset?

Splitting your dataset is a fundamental practice in machine learning for several reasons:

1. **Preventing Overfitting**: Training and evaluating on the same data leads to models that memorize the training data rather than generalizing to new data.

2. **Reliable Performance Estimation**: A separate test set provides an unbiased evaluation of model performance on unseen data.

3. **Model Selection**: A validation set allows you to tune hyperparameters without contaminating your test set.

4. **Fair Comparison**: Standardized splits enable fair comparisons between different models or approaches.

Let's see how to implement these concepts in practice.

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# For dataset splitting
from sklearn.model_selection import train_test_split, KFold, cross_val_score, StratifiedKFold

# For modeling
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# For datasets
from sklearn import datasets

# Set a random seed for reproducibility
np.random.seed(42)

## 1. Basic Train-Test Split

The simplest approach is to split your dataset into two parts:
- **Training set**: Used to train the model (typically 70-80% of the data)
- **Test set**: Used to evaluate the model's performance (typically 20-30% of the data)

Let's use the Iris dataset as our first example:

In [ ]:
# Load the Iris dataset
iris = datasets.load_iris()
X = iris.data  # Features
y = iris.target  # Target variable

# Let's examine the dataset
print(f"Dataset shape: {X.shape}")
print(f"Number of classes: {len(np.unique(y))}")
print(f"Class distribution: {np.bincount(y)}")

# Create a DataFrame for better visualization
iris_df = pd.DataFrame(data=np.c_[iris['data'], iris['target']],
                      columns=iris['feature_names'] + ['target'])
iris_df['species'] = iris_df['target'].map({
    0: 'setosa',
    1: 'versicolor',
    2: 'virginica'
})

# Display the first few rows
iris_df.head()

In [ ]:
# Basic train-test split (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Print the shapes of the resulting datasets
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

# Check class distribution in training and test sets
print(f"Training set class distribution: {np.bincount(y_train)}")
print(f"Test set class distribution: {np.bincount(y_test)}")

### Training and Evaluating a Model with Train-Test Split

Now let's train a simple model using our train-test split and evaluate its performance:

In [ ]:
# Create a Random Forest classifier
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the model on the training data
rf_classifier.fit(X_train, y_train)

# Make predictions on the test data
y_pred = rf_classifier.predict(X_test)

# Evaluate the model's performance
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

# Generate a more detailed classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

# Create a confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=iris.target_names, 
            yticklabels=iris.target_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

## 2. Train-Validation-Test Split

For more complex models that require hyperparameter tuning, we typically split the data into three parts:

- **Training set**: Used to train the model (typically 60-70% of the data)
- **Validation set**: Used for hyperparameter tuning and model selection (typically 15-20% of the data)
- **Test set**: Used for final evaluation of the selected model (typically 15-20% of the data)

This approach helps prevent information leakage from the test set during model selection.

In [ ]:
# First, split the data into a temporary train set and the test set
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Then, split the temporary set into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42  # 0.25 of 80% = 20% of original data
)

# Print the shapes of the resulting datasets
print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"X_test shape: {X_test.shape}")

# Check class distribution
print(f"Training set class distribution: {np.bincount(y_train)}")
print(f"Validation set class distribution: {np.bincount(y_val)}")
print(f"Test set class distribution: {np.bincount(y_test)}")

### Using the Validation Set for Hyperparameter Tuning

Let's use our validation set to select the best hyperparameters for a Random Forest model:

In [ ]:
# Let's try different values for n_estimators (number of trees in the forest)
n_estimators_options = [10, 50, 100, 200]
best_accuracy = 0
best_n_estimators = None

for n_estimators in n_estimators_options:
    # Create and train the model
    rf = RandomForestClassifier(n_estimators=n_estimators, random_state=42)
    rf.fit(X_train, y_train)
    
    # Evaluate on validation set
    val_predictions = rf.predict(X_val)
    val_accuracy = accuracy_score(y_val, val_predictions)
    
    print(f"n_estimators={n_estimators}, Validation Accuracy: {val_accuracy:.4f}")
    
    # Track best parameters
    if val_accuracy > best_accuracy:
        best_accuracy = val_accuracy
        best_n_estimators = n_estimators

print(f"\nBest n_estimators: {best_n_estimators}")
print(f"Best validation accuracy: {best_accuracy:.4f}")

In [ ]:
# Train final model with best parameters on combined training and validation data
X_train_val = np.vstack((X_train, X_val))
y_train_val = np.hstack((y_train, y_val))

final_model = RandomForestClassifier(n_estimators=best_n_estimators, random_state=42)
final_model.fit(X_train_val, y_train_val)

# Evaluate on the test set
test_predictions = final_model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)

print(f"Final test accuracy: {test_accuracy:.4f}")
print("\nClassification Report on Test Set:")
print(classification_report(y_test, test_predictions, target_names=iris.target_names))

## 3. K-Fold Cross-Validation

K-fold cross-validation is a technique that helps make better use of the available data, especially when the dataset is small.

### Why Use K-Fold Cross-Validation?

1. **Efficient use of data**: Every data point gets used for both training and validation.
   
2. **Robust performance estimation**: Results are less dependent on a particular split of the data.
   
3. **Reduced variance**: The performance estimate is averaged over multiple train-validation splits.
   
4. **Better hyperparameter tuning**: More reliable estimates of model performance for different hyperparameters.

### How K-Fold Cross-Validation Works:

1. The dataset is divided into K equal-sized folds (typically 5 or 10).
2. The model is trained K times, each time using K-1 folds for training and the remaining fold for validation.
3. The K validation results are averaged to produce a single performance estimate.

Let's implement K-fold cross-validation:

In [ ]:
# Create a model
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Set up 5-fold cross-validation
k_folds = 5
kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)

# Lists to store results from each fold
fold_accuracies = []

# Perform k-fold cross-validation
for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    # Split data for this fold
    X_train_fold, X_val_fold = X[train_idx], X[val_idx]
    y_train_fold, y_val_fold = y[train_idx], y[val_idx]
    
    # Train the model
    model.fit(X_train_fold, y_train_fold)
    
    # Evaluate on validation data
    val_predictions = model.predict(X_val_fold)
    fold_accuracy = accuracy_score(y_val_fold, val_predictions)
    fold_accuracies.append(fold_accuracy)
    
    print(f"Fold {fold+1} Accuracy: {fold_accuracy:.4f}")

# Calculate average accuracy across all folds
avg_accuracy = np.mean(fold_accuracies)
std_accuracy = np.std(fold_accuracies)

print(f"\nAverage Accuracy: {avg_accuracy:.4f} ± {std_accuracy:.4f}")

### Using scikit-learn's Built-in Cross-Validation Function

Scikit-learn provides a simpler way to perform cross-validation:

In [ ]:
# Use cross_val_score for a more concise implementation
model = RandomForestClassifier(n_estimators=100, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=5)

print("Individual fold scores:")
for i, score in enumerate(cv_scores):
    print(f"Fold {i+1}: {score:.4f}")

print(f"\nAverage CV Score: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")

### Hyperparameter Tuning with Cross-Validation

We can combine cross-validation with hyperparameter tuning to find the best model parameters:

In [ ]:
# Hyperparameter tuning with cross-validation
n_estimators_options = [10, 50, 100, 200]
max_depth_options = [None, 5, 10]

best_cv_score = 0
best_params = {}

for n_estimators in n_estimators_options:
    for max_depth in max_depth_options:
        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=42
        )
        
        # Perform cross-validation
        cv_scores = cross_val_score(model, X, y, cv=5)
        avg_cv_score = np.mean(cv_scores)
        
        print(f"n_estimators={n_estimators}, max_depth={max_depth}: CV Score={avg_cv_score:.4f}")
        
        # Track best parameters
        if avg_cv_score > best_cv_score:
            best_cv_score = avg_cv_score
            best_params = {'n_estimators': n_estimators, 'max_depth': max_depth}

print(f"\nBest Parameters: {best_params}")
print(f"Best CV Score: {best_cv_score:.4f}")

## 4. Handling Imbalanced Datasets

In many real-world scenarios, class distributions are not equal (imbalanced). This can lead to biased models that favor the majority class. Let's explore how to handle imbalanced datasets using:

1. Stratified sampling
2. Class weights
3. Specialized evaluation metrics

For this example, we'll use the breast cancer dataset, which has a slight class imbalance:

In [ ]:
# Load the breast cancer dataset
cancer = datasets.load_breast_cancer()
X_cancer = cancer.data
y_cancer = cancer.target

# Check class distribution
unique, counts = np.unique(y_cancer, return_counts=True)
class_distribution = dict(zip(unique, counts))

print("Class distribution:")
for class_label, count in class_distribution.items():
    class_name = "Malignant" if class_label == 0 else "Benign"
    percentage = count / len(y_cancer) * 100
    print(f"Class {class_label} ({class_name}): {count} samples ({percentage:.2f}%)")

# Visualize the class distribution
plt.figure(figsize=(10, 6))
sns.countplot(x=y_cancer)
plt.xticks([0, 1], ['Malignant (0)', 'Benign (1)'])
plt.title('Class Distribution in Breast Cancer Dataset')
plt.xlabel('Class')
plt.ylabel('Count')
plt.show()

### 4.1 Stratified Sampling

Stratified sampling ensures that the class distribution in your train and test sets matches the original distribution. This is particularly important for imbalanced datasets.

In [ ]:
# Regular train-test split (without stratification)
X_train_regular, X_test_regular, y_train_regular, y_test_regular = train_test_split(
    X_cancer, y_cancer, test_size=0.3, random_state=42
)

# Stratified train-test split
X_train_strat, X_test_strat, y_train_strat, y_test_strat = train_test_split(
    X_cancer, y_cancer, test_size=0.3, random_state=42, stratify=y_cancer
)

# Compare class distributions
print("Original dataset class distribution:")
print(f"Class 0 (Malignant): {np.sum(y_cancer == 0) / len(y_cancer):.4f}")
print(f"Class 1 (Benign): {np.sum(y_cancer == 1) / len(y_cancer):.4f}")

print("\nRegular split - Training set class distribution:")
print(f"Class 0 (Malignant): {np.sum(y_train_regular == 0) / len(y_train_regular):.4f}")
print(f"Class 1 (Benign): {np.sum(y_train_regular == 1) / len(y_train_regular):.4f}")

print("\nRegular split - Test set class distribution:")
print(f"Class 0 (Malignant): {np.sum(y_test_regular == 0) / len(y_test_regular):.4f}")
print(f"Class 1 (Benign): {np.sum(y_test_regular == 1) / len(y_test_regular):.4f}")

print("\nStratified split - Training set class distribution:")
print(f"Class 0 (Malignant): {np.sum(y_train_strat == 0) / len(y_train_strat):.4f}")
print(f"Class 1 (Benign): {np.sum(y_train_strat == 1) / len(y_train_strat):.4f}")

print("\nStratified split - Test set class distribution:")
print(f"Class 0 (Malignant): {np.sum(y_test_strat == 0) / len(y_test_strat):.4f}")
print(f"Class 1 (Benign): {np.sum(y_test_strat == 1) / len(y_test_strat):.4f}")

### 4.2 Class Weights

When dealing with imbalanced datasets, we can use class weights to give more importance to the minority class during training:

In [ ]:
# Using class weights with LogisticRegression
# Train a model without class weights
model_no_weights = LogisticRegression(random_state=42)
model_no_weights.fit(X_train_strat, y_train_strat)
y_pred_no_weights = model_no_weights.predict(X_test_strat)

# Train a model with balanced class weights
model_with_weights = LogisticRegression(class_weight='balanced', random_state=42)
model_with_weights.fit(X_train_strat, y_train_strat)
y_pred_with_weights = model_with_weights.predict(X_test_strat)

# Compare results
print("Model without class weights:")
print(classification_report(y_test_strat, y_pred_no_weights, 
                           target_names=['Malignant', 'Benign']))

print("\nModel with balanced class weights:")
print(classification_report(y_test_strat, y_pred_with_weights, 
                           target_names=['Malignant', 'Benign']))

# Visualize confusion matrices
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Confusion matrix for model without weights
cm1 = confusion_matrix(y_test_strat, y_pred_no_weights)
sns.heatmap(cm1, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['Malignant', 'Benign'],
            yticklabels=['Malignant', 'Benign'])
ax1.set_xlabel('Predicted')
ax1.set_ylabel('True')
ax1.set_title('Confusion Matrix - No Class Weights')

# Confusion matrix for model with weights
cm2 = confusion_matrix(y_test_strat, y_pred_with_weights)
sns.heatmap(cm2, annot=True, fmt='d', cmap='Blues', ax=ax2,
            xticklabels=['Malignant', 'Benign'],
            yticklabels=['Malignant', 'Benign'])
ax2.set_xlabel('Predicted')
ax2.set_ylabel('True')
ax2.set_title('Confusion Matrix - With Class Weights')

plt.tight_layout()
plt.show()

### 4.3 Stratified K-Fold Cross-Validation

For imbalanced datasets, it's important to use stratified K-fold cross-validation to maintain the class distribution across all folds:

In [ ]:
# Regular K-Fold vs Stratified K-Fold
model = LogisticRegression(random_state=42)

# Regular K-Fold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
regular_cv_scores = cross_val_score(model, X_cancer, y_cancer, cv=kf)

# Stratified K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
stratified_cv_scores = cross_val_score(model, X_cancer, y_cancer, cv=skf)

print("Regular K-Fold CV Scores:", regular_cv_scores)
print(f"Average: {np.mean(regular_cv_scores):.4f} ± {np.std(regular_cv_scores):.4f}")

print("\nStratified K-Fold CV Scores:", stratified_cv_scores)
print(f"Average: {np.mean(stratified_cv_scores):.4f} ± {np.std(stratified_cv_scores):.4f}")

# Visualize the scores
plt.figure(figsize=(10, 6))
plt.plot(range(1, 6), regular_cv_scores, 'o-', label='Regular K-Fold')
plt.plot(range(1, 6), stratified_cv_scores, 'o-', label='Stratified K-Fold')
plt.xlabel('Fold')
plt.ylabel('Accuracy')
plt.title('Regular vs Stratified K-Fold Cross-Validation')
plt.legend()
plt.grid(True)
plt.show()

## 5. Creating a More Severely Imbalanced Dataset Example

To demonstrate techniques for highly imbalanced datasets, let's create a more imbalanced version of our dataset:

In [ ]:
# Let's create a more severely imbalanced dataset
from sklearn.datasets import make_classification

# Generate an imbalanced dataset
X_imb, y_imb = make_classification(
    n_samples=1000, 
    n_features=20,
    n_informative=15,
    n_redundant=5,
    n_classes=2,
    weights=[0.9, 0.1],  # 90% class 0, 10% class 1
    random_state=42
)

# Check class distribution
unique, counts = np.unique(y_imb, return_counts=True)
class_distribution = dict(zip(unique, counts))

print("Class distribution in synthetic imbalanced dataset:")
for class_label, count in class_distribution.items():
    percentage = count / len(y_imb) * 100
    print(f"Class {class_label}: {count} samples ({percentage:.2f}%)")

# Visualize the class distribution
plt.figure(figsize=(10, 6))
sns.countplot(x=y_imb)
plt.title('Class Distribution in Synthetic Imbalanced Dataset')
plt.xlabel('Class')
plt.ylabel('Count')
plt.show()

### 5.1 Using SMOTE for Imbalanced Datasets

SMOTE (Synthetic Minority Over-sampling Technique) is a popular technique for handling imbalanced datasets by generating synthetic samples for the minority class:

In [ ]:
# Install imbalanced-learn if not already installed
# !pip install imbalanced-learn

# Import SMOTE
from imblearn.over_sampling import SMOTE

# Split the imbalanced dataset
X_train_imb, X_test_imb, y_train_imb, y_test_imb = train_test_split(
    X_imb, y_imb, test_size=0.3, random_state=42, stratify=y_imb
)

# Check training set class distribution before SMOTE
print("Class distribution before SMOTE:")
for class_label, count in zip(*np.unique(y_train_imb, return_counts=True)):
    percentage = count / len(y_train_imb) * 100
    print(f"Class {class_label}: {count} samples ({percentage:.2f}%)")

# Apply SMOTE to the training data
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_imb, y_train_imb)

# Check training set class distribution after SMOTE
print("\nClass distribution after SMOTE:")
for class_label, count in zip(*np.unique(y_train_smote, return_counts=True)):
    percentage = count / len(y_train_smote) * 100
    print(f"Class {class_label}: {count} samples ({percentage:.2f}%)")

# Train models with and without SMOTE
model_no_smote = LogisticRegression(random_state=42)
model_no_smote.fit(X_train_imb, y_train_imb)
y_pred_no_smote = model_no_smote.predict(X_test_imb)

model_with_smote = LogisticRegression(random_state=42)
model_with_smote.fit(X_train_smote, y_train_smote)
y_pred_with_smote = model_with_smote.predict(X_test_imb)

# Compare results
print("\nModel without SMOTE:")
print(classification_report(y_test_imb, y_pred_no_smote))

print("\nModel with SMOTE:")
print(classification_report(y_test_imb, y_pred_with_smote))

## 6. Summary and Best Practices

Here's a summary of best practices for dataset splitting in machine learning:

1. **Always split your data** to get an honest evaluation of model performance.

2. **Use stratified sampling** when dealing with classification problems to maintain the same class distribution in train and test sets.

3. **Consider a three-way split** (train-validation-test) for complex models that require hyperparameter tuning.

4. **Use cross-validation** for small datasets or when you need a more robust performance estimate.

5. **For imbalanced datasets**:
   - Use stratified sampling to maintain class distribution
   - Consider using class weights during training
   - Apply techniques like SMOTE to generate synthetic samples for minority classes
   - Use appropriate evaluation metrics (precision, recall, F1-score) instead of just accuracy

6. **Report performance metrics** on the test set only after all model development is complete.

7. **Ensure reproducibility** by setting random seeds consistently across your workflow.

By following these practices, you'll develop more robust machine learning models that generalize well to new, unseen data.